# Liquid level detection (LLD)

Liquid level detection (LLD) is a feature that allows the Hamilton STAR(let) to move the pipetting tip down slowly until a liquid is found using either a) the pressure sensor, b) a change in capacitance, or c) both. This is useful when you want to aspirate or dispense at a distance relative to the liquid surface, but you don't know the exact height of the liquid in the container.

The LLD mode is specified via {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.LLDMode`, passed as a backend parameter to {meth}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.aspirate` or {meth}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.dispense`.

## Setup

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.hamilton.liquid_handlers.star.pip_backend import STARPIPBackend, LLDMode
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

Set up a minimal deck with a tip rack and a tube rack (or plate) so we have something to aspirate from.

In [ ]:
from pylabrobot.resources import (
  TIP_CAR_480_A00,
  PLT_CAR_L5AC_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
deck.assign_child_resource(plt_car, rails=15)

tiprack = deck.get_resource("tips_01")
plate = deck.get_resource("plate_01")

Pick up tips so we can demonstrate aspiration and dispensing with LLD.

In [ ]:
await star.pip.pick_up_tips(tiprack["A1"])

## LLD modes

To use LLD, pass a {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.LLDMode` list to `lld_mode` inside {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams` or {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.DispenseParams`.

The available modes are:

| Mode | Description |
|------|-------------|
| `LLDMode.OFF` | No LLD (default) |
| `LLDMode.GAMMA` | Capacitive LLD (cLLD) |
| `LLDMode.PRESSURE` | Pressure LLD (pLLD) |
| `LLDMode.DUAL` | Both capacitive and pressure LLD |
| `LLDMode.Z_TOUCH_OFF` | Find the bottom of the container |

The `lld_mode` parameter is a list, so you can specify a different LLD mode for each channel.

In [ ]:
await star.pip.aspirate(
  plate["A1"],
  vols=[300],
  backend_params=STARPIPBackend.AspirateParams(
    lld_mode=[LLDMode.GAMMA],
  ),
)

## Immersion depth

You can use the `immersion_depth` field on {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams` to move the tip relative to the detected liquid surface. A positive value means to go deeper into the liquid; a negative value means to stay above the liquid.

Going 1 mm below the liquid surface for aspiration:

In [ ]:
await star.pip.aspirate(
  plate["A1"],
  vols=[300],
  backend_params=STARPIPBackend.AspirateParams(
    lld_mode=[LLDMode.GAMMA],
    immersion_depth=[1],
  ),
)

Going 1 mm above the liquid surface for dispensing:

In [ ]:
await star.pip.dispense(
  plate["A1"],
  vols=[300],
  backend_params=STARPIPBackend.DispenseParams(
    lld_mode=[LLDMode.GAMMA],
    immersion_depth=[-1],
  ),
)

## Surface following

Through the `surface_following_distance` field on {class}`~pylabrobot.hamilton.liquid_handlers.star.pip_backend.STARPIPBackend.AspirateParams`, the tip can track the falling liquid surface during aspiration:

In [ ]:
await star.pip.aspirate(
  plate["A1"],
  vols=[300],
  backend_params=STARPIPBackend.AspirateParams(
    lld_mode=[LLDMode.GAMMA],
    surface_following_distance=[10],  # 10 mm
  ),
)

## Error handling

All channelized pipetting operations raise a {class}`~pylabrobot.capabilities.liquid_handling.errors.ChannelizedError` when an error occurs, so that you get specific error information for each channel.

When no liquid is found in the container, the channel will have a {class}`~pylabrobot.resources.errors.TooLittleLiquidError`. This is useful for detecting that your container is empty.

In [ ]:
from pylabrobot.capabilities.liquid_handling.errors import ChannelizedError
from pylabrobot.resources.errors import TooLittleLiquidError

try:
  await star.pip.aspirate(
    plate["A1"],
    vols=[300],
    backend_params=STARPIPBackend.AspirateParams(
      lld_mode=[LLDMode.GAMMA],
    ),
  )
except ChannelizedError as e:
  if isinstance(e.errors[0], TooLittleLiquidError):
    print("Too little liquid in well")

## Teardown

In [ ]:
await star.pip.drop_tips(tiprack["A1"])
await star.stop()